In [ ]:
# Import Dependencies

# Standard Imports
import json
from typing import Any

# Third-party Imports
import httpx

# Local modules
from lib.fhir_client import FHIR_BASE_URL, HEADERS

print(f"FHIR_BASE_URL: {FHIR_BASE_URL}")
print(f"HEADERS: {HEADERS}")

In [ ]:
# Fetch CapabilityStatement

def fetch_capability_statement() -> dict[str, Any] | None:
    # httpx.Client is used as a context manager: the 'with' block opens a
    # connection pool on entry and closes it cleanly on exit (even on error).
    try:
        with httpx.Client() as client:
            response = client.get(f"{FHIR_BASE_URL}/metadata", headers=HEADERS, timeout=10)
            response.raise_for_status()
            return response.json()
    except httpx.ConnectError:
        print(f"Could not connect to FHIR server at {FHIR_BASE_URL}.")
        print("Is the HAPI FHIR server running?")
        return None
    except httpx.TimeoutException:
        print(f"FHIR server at {FHIR_BASE_URL} timed out.")
        return None
    except httpx.DecodingError:
        print("FHIR server response was not valid JSON.")
        return None

In [ ]:
# Print Capability Statement

capability_statement = fetch_capability_statement()

if capability_statement is not None:
    print(json.dumps(capability_statement, indent=2))
else:
    print("capability_statement is None — check server connection.")

In [ ]:
# Print CapabilityStatement Summary

def print_capability_summary(capability_statement: dict[str, Any]) -> None:
    # Pull a few top-level fields out of the CapabilityStatement for easier reading
    resource_type = capability_statement.get("resourceType", "Unknown")
    capability_id = capability_statement.get("id", "Unknown")
    fhir_version = capability_statement.get("fhirVersion", "Unknown")

    # The implementation field is a nested dictionary, so read it separately first.
    implementation = capability_statement.get("implementation", {})

    # Now you can pull items from implementation
    implementation_description = implementation.get("description", "Unknown")
    implementation_url = implementation.get("url", "Unknown")

    print("-" * 60)
    print("CapabilityStatement Summary")
    print("-" * 60)
    print(f"             Resource Type: {resource_type}")
    print(f"                        ID: {capability_id}")
    print(f"Implementation Description: {implementation_description}")
    print(f"        Implementation URL: {implementation_url}")
    print(f"              FHIR Version: {fhir_version}")
    print()

print_capability_summary(capability_statement)

In [ ]:
# Extract a list of supported resources

def get_resource_list(capability_statement: dict[str, Any]) -> list[str]:
    resource_list: list[str] = []

    # CapabilityStatement.rest is a list; each rest entry can advertise many resources
    for rest_entry in capability_statement.get("rest", []):
        for resource in rest_entry.get("resource", []):
            resource_type = resource.get("type")
            if resource_type:
                resource_list.append(resource_type)

    return resource_list

In [ ]:
# Print Resource List

resource_list = get_resource_list(capability_statement)

print("-" * 60)
print("Supported FHIR Resources")
print("-" * 60)
print(f"Resource Count: {len(resource_list)}")
print()

for i, resource in enumerate(resource_list, start=1):
    if i % 3 == 0:
        print(resource)             # third column: let print() supply the newline
    else:
        print(f"{resource:<35}", end='')   # first/second column: suppress newline

print()

In [ ]:
# Show all search parameters for a given Resource

resource_type = "Account"

rest_list = capability_statement.get("rest", [])

# Guard against an empty rest list before indexing with [0].
rest_entry = rest_list[0] if rest_list else {}

# resource_entries is a list of dicts — one dict per FHIR resource type.
resource_entries = rest_entry.get("resource", [])

# Select the resource entry whose "type" matches resource_type
resource_entry = next(
    (r for r in resource_entries if r.get("type") == resource_type),
    None  # default: returned when resource_type is not found in the list
)

if resource_entry is None:
    print(f"Resource type '{resource_type}' not found in CapabilityStatement.")
else:
    # searchParam is a list of dicts, each describing one supported search parameter.
    search_params = resource_entry.get("searchParam", [])

    print("-" * 60)
    print(f"Search Parameters for: {resource_type}")
    print("-" * 60)
    print(f"Count: {len(search_params)}")
    print()

    # Each param dict has at minimum "name" and "type"; "definition" (a URL) is
    # also common but not always present, so .get() with a fallback is safer.
    for param in search_params:
        name = param.get("name", "(unknown)")
        param_type = param.get("type", "(unknown)")
        print(f"  {name:<35} type: {param_type}")